# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python) 

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.

**Instrucciones:**

- Importa las librerías necesarias
- Carga los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Guarda los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Explora cada dataset.

---

In [1]:
# importar librerías
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [2]:
# cargar archivos
orders = pd.read_csv('rappiplus_orders_raw.csv')
catalog = pd.read_csv('rappiplus_catalog.csv')
marketing = pd.read_csv('rappiplus_marketing_spend.csv')

In [3]:
# explorar datasets
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  object 
 1   id_usuario          25100 non-null  object 
 2   fecha_hora_pedido   25100 non-null  object 
 3   pais                24800 non-null  object 
 4   dispositivo         25080 non-null  object 
 5   fuente_referencia   25070 non-null  object 
 6   nombre_producto     25070 non-null  object 
 7   categoria_producto  25020 non-null  object 
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), object(8)
memory usage: 2.3+ MB


In [4]:
catalog.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      object 
 1   categoria_producto  7 non-null      object 
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      object 
dtypes: float64(1), object(3)
memory usage: 352.0+ bytes


In [5]:
marketing.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   fecha       1620 non-null   object 
 1   pais        1620 non-null   object 
 2   id_campaña  1620 non-null   object 
 3   canal       1519 non-null   object 
 4   gasto       1620 non-null   float64
dtypes: float64(1), object(4)
memory usage: 63.4+ KB


---

### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas 

---

In [6]:
# Conversión a tipo fecha de "fecha_hora_pedido" en orders
orders["fecha_hora_pedido"] = pd.to_datetime(orders['fecha_hora_pedido'], errors='coerce')
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id_pedido           25100 non-null  object        
 1   id_usuario          25100 non-null  object        
 2   fecha_hora_pedido   25100 non-null  datetime64[ns]
 3   pais                24800 non-null  object        
 4   dispositivo         25080 non-null  object        
 5   fuente_referencia   25070 non-null  object        
 6   nombre_producto     25070 non-null  object        
 7   categoria_producto  25020 non-null  object        
 8   cantidad            25050 non-null  float64       
 9   precio_unitario     25050 non-null  float64       
 10  monto_descuento     25050 non-null  float64       
 11  monto_total         25100 non-null  float64       
dtypes: datetime64[ns](1), float64(4), object(7)
memory usage: 2.3+ MB


In [7]:
# Conversión de cantidades negativas a positivas en orders y verificación
orders["cantidad"] = orders["cantidad"].abs()
orders["cantidad"].describe()

count    25050.000000
mean         7.093134
std        296.276993
min          1.000000
25%          1.000000
50%          2.000000
75%          2.000000
max      20000.000000
Name: cantidad, dtype: float64

In [8]:
# Conversión de montos negativos a positivos en orders y verificación
orders["monto_total"] = orders["monto_total"].abs()
orders["monto_total"].describe()

count    2.510000e+04
mean     2.072771e+03
std      9.894995e+04
min      5.240000e+00
25%      1.805925e+02
50%      3.418050e+02
75%      5.185800e+02
max      8.840200e+06
Name: monto_total, dtype: float64

In [9]:
# Eliminación de duplicados y verificación
orders = orders.drop_duplicates(subset='id_pedido')
orders.duplicated().sum()

0

In [10]:
# Conversión de "fecha" a tipo fecha en marketing
marketing["fecha"] = pd.to_datetime(marketing['fecha'], errors='coerce')
marketing.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   fecha       1620 non-null   datetime64[ns]
 1   pais        1620 non-null   object        
 2   id_campaña  1620 non-null   object        
 3   canal       1519 non-null   object        
 4   gasto       1620 non-null   float64       
dtypes: datetime64[ns](1), float64(1), object(3)
memory usage: 63.4+ KB


In [11]:
# Conversión de datos vacíos a NaN en marketing
marketing["canal"] = marketing["canal"].replace("", np.nan)
marketing.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   fecha       1620 non-null   datetime64[ns]
 1   pais        1620 non-null   object        
 2   id_campaña  1620 non-null   object        
 3   canal       1519 non-null   object        
 4   gasto       1620 non-null   float64       
dtypes: datetime64[ns](1), float64(1), object(3)
memory usage: 63.4+ KB


In [12]:
df_merged = pd.merge(orders, catalog, on='nombre_producto',how='left')
df_merged.isna().sum()

id_pedido                 0
id_usuario                0
fecha_hora_pedido         0
pais                    300
dispositivo              20
fuente_referencia        30
nombre_producto          30
categoria_producto_x     80
cantidad                 50
precio_unitario          50
monto_descuento          50
monto_total               0
categoria_producto_y     30
costo_unitario           30
proveedor                30
dtype: int64

In [13]:
# Revisión de nombre_producto
orders["nombre_producto"].value_counts(dropna=False)

Blender-XL-Red          4179
Jacket-Winter-M         4176
Vacuum-Pro-Black        4176
Sneakers-Urban-42       4148
Laptop-Gaming-16GB      2782
Tablet-Standard-64GB    2769
Phone-Pro-128GB         2740
NaN                       30
Name: nombre_producto, dtype: int64

In [14]:
# Limpieza y verificación de nombres en 'pais'
orders["pais"] = orders["pais"].str.title()
orders["pais"].value_counts()

Mexico       8341
Colombia     8304
Argentina    8055
Name: pais, dtype: int64

In [15]:
# Verificación de dispositivo
orders["dispositivo"].value_counts()

desktop    12711
mobile     12269
Name: dispositivo, dtype: int64

El dispositivo 'desktop' es el más utilizado para hacer uso del servicio con una diferencia de 442 pedidos.

In [16]:
# Verificación de fuente_referencia
orders["fuente_referencia"].value_counts()

social         8402
organic        8288
paid_search    8280
Name: fuente_referencia, dtype: int64

La fuente principal de referencia es 'social' lo que indica un mejor desempeño de las redes sociales para atraer clientes pero sin haber mucha diferencia para recomendar que la inversion mayoritaria vaya a la fuente 'social'.

In [17]:
# Verificación de categoria_producto
orders["categoria_producto"].value_counts()

Hogar          8346
Moda           8295
Electronica    8279
Name: categoria_producto, dtype: int64

Los productos de la categoría 'Hogar' son los más vendidos por una pequeña diferencia respecto a las demás categorías.

In [18]:
# Verificación de cantidad
orders["cantidad"].value_counts()

2.0        12592
1.0        12348
10000.0        6
20000.0        4
Name: cantidad, dtype: int64

En la columna 'cantidad', encontramos posibles outliers pues en 11 de las 25,100 filas hay valores atípicos en los que se observan pedidos mucho mayores al promedio, sin embargo no se eliminan al no poder verificar la veracidad de los datos evitando dañar la integridad del dataset.

---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

In [19]:
# exportar datasets
orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)

---

## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)? 
- ¿Cuál es el costo total? 
- ¿Cuánto se ha invertido en marketing? 
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden? 
- ¿Cuál es la cantidad promedio de productos por orden? 
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal? 

In [20]:
# Parte 1
# Ingreso Total
ingreso_total = orders["monto_total"].sum()
print(f"Ingreso Total:\n{ingreso_total:,.2f}")

Ingreso Total:
51,989,756.86


In [21]:
# Costo Total
costo_total = (df_merged['cantidad'] * df_merged["costo_unitario"]).sum()
print(f"Costo Total:\n{costo_total:,.2f}")

Costo Total:
43,124,119.61


In [22]:
# Inversión en marketing
inversion_marketing = marketing["gasto"].sum()
print(f"Inversion En Marketing:\n{inversion_marketing:,.2f}")

Inversion En Marketing:
2,871,843.53


La inversión total incluye 101 filas de gastos en los que faltan los datos del canal de marketing dando un total de 177,179.10 siendo una suma importante por lo que se decidió no eliminar las filas.

In [23]:
# Rentabilidad (Profit)
rentabilidad = ingreso_total - (costo_total + inversion_marketing)
print(f"Rentabilidad:\n{rentabilidad:,.2f}")

Rentabilidad:
5,993,793.72


---

In [24]:
# Parte 2

# Ticket Promedio
ticket_promedio = orders["monto_total"].mean()
print(f"Ticket Promedio:\n{ticket_promedio:,.2f}")

Ticket Promedio:
2,079.59


In [25]:
# Cantidad Promedio por orden
cantidad_promedio = orders["cantidad"].mean()
print(f"Cantidad Promedio:\n{cantidad_promedio:.2f}")

Cantidad Promedio:
7.12


In [26]:
# Producto más vendido
orders["nombre_producto"].value_counts().index[0]

'Blender-XL-Red'

In [27]:
# Gasto en marketing por canal
gasto_marketing = marketing.groupby('canal')['gasto'].sum()
print(f"Gasto en marketing por canal:\n{gasto_marketing}")
print()
print(f"Total:\n{gasto_marketing.sum():,.2f}")

Gasto en marketing por canal:
canal
organic        913533.01
paid_search    863088.21
social         918043.21
Name: gasto, dtype: float64

Total:
2,694,664.43


Aquí no se están tomando en cuenta las filas con NaN de la columna 'canal' por lo que el gasto total representa únicamente la suma de los canales registrados.
(Para ver el gasto total revisar inversion_marketing definido arriba).

---

## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [28]:
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [29]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [30]:
# PARTE 1: Totales del funnel
# ======================

query_totals = '''
SELECT nombre_evento, COUNT (DISTINCT id_usuario)
FROM events
GROUP BY nombre_evento
ORDER BY CASE nombre_evento
    WHEN 'first_visit' THEN 1
    WHEN 'select_item' THEN 2
    WHEN 'add_to_cart' THEN 3
    WHEN 'begin_checkout' THEN 4
    WHEN 'add_payment_info' THEN 5
    WHEN 'purchase' THEN 6
END;
'''

totals = pd.read_sql(query_totals, con=engine)
totals

,nombre_evento,count
0,first_visit,7796
1,select_item,7582
2,add_to_cart,7634
3,begin_checkout,7208
4,add_payment_info,6250
5,purchase,6240


In [31]:
# PARTE 2: Conversiones
# ======================

query_conversion = '''
WITH cte_first AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'first_visit'
),
cte_select AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'select_item'  
),
cte_cart AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'add_to_cart'  
),
cte_checkout AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'begin_checkout'  
),
cte_payment AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'add_payment_info'  
),
cte_purchase AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'purchase'  
)
SELECT
    (SELECT COUNT(*) FROM cte_first) AS first_users,
    (SELECT COUNT(*) FROM cte_select) AS select_users,
    (SELECT COUNT(*) FROM cte_cart) AS cart_users,
    (SELECT COUNT(*) FROM cte_checkout) AS checkout_users,
    (SELECT COUNT(*) FROM cte_payment) AS payment_users,
    (SELECT COUNT(*) FROM cte_purchase) AS purchase_users,
        ((SELECT COUNT(*) FROM cte_select)) * 100
        / NULLIF((SELECT COUNT(*) FROM cte_first), 0) AS conversion_after_first_visit,
        ((SELECT COUNT(*) FROM cte_cart)) * 100
        / NULLIF((SELECT COUNT(*) FROM cte_select), 0) AS conversion_after_select_item,
        ((SELECT COUNT(*) FROM cte_checkout)) * 100
        / NULLIF((SELECT COUNT(*) FROM cte_cart), 0) AS conversion_after_add_to_cart,
        ((SELECT COUNT(*) FROM cte_payment)) * 100
        / NULLIF((SELECT COUNT(*) FROM cte_checkout), 0) AS conversion_after_checkout,
        ((SELECT COUNT(*) FROM cte_purchase)) * 100
        / NULLIF((SELECT COUNT(*) FROM cte_payment), 0) AS conversion_after_payment;
    '''


conversion = pd.read_sql(query_conversion, con=engine)
conversion

,first_users,select_users,cart_users,checkout_users,payment_users,purchase_users,conversion_after_first_visit,conversion_after_select_item,conversion_after_add_to_cart,conversion_after_checkout,conversion_after_payment
0,7796,7582,7634,7208,6250,6240,97,100,94,86,99


---

## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users` 
- `user_activity` 

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [32]:
# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


In [33]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
SELECT *
FROM user_activity;
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)

,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1


In [34]:
# Revisión de tipos de datos en user_activity
user_activity.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32000 entries, 0 to 31999
Data columns (total 4 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   id_usuario             32000 non-null  object
 1   fecha_actividad        32000 non-null  object
 2   dias_despues_registro  32000 non-null  int64 
 3   activo                 32000 non-null  int64 
dtypes: int64(2), object(2)
memory usage: 1000.1+ KB


In [35]:
# Retención por cohortes
# ======================
# Retención Semanal

query_cohort_retention_final = '''
WITH cohortes AS (
    SELECT
        id_usuario,
        DATE_TRUNC('week', MIN(CAST(fecha_actividad AS timestamp))) AS cohorte_semana,
        dias_despues_registro,
        activo
    FROM user_activity
    GROUP BY id_usuario, dias_despues_registro, activo
)
SELECT 
    cohorte_semana,
    COUNT(*) AS usuarios_iniciales,
    COUNT(CASE WHEN dias_despues_registro >= 7 AND activo = 1 THEN 1 END) AS retenido_w1,
    COUNT(CASE WHEN dias_despues_registro >= 14 AND activo = 1 THEN 1 END) AS retenido_w2,
    COUNT(CASE WHEN dias_despues_registro >= 21 AND activo = 1 THEN 1 END) AS retenido_w3
FROM cohortes
GROUP BY cohorte_semana
'''

# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final

,cohorte_semana,usuarios_iniciales,retenido_w1,retenido_w2,retenido_w3
0,2025-06-02,1465,605,463,309
1,2025-03-10,1418,564,420,272
2,2025-03-24,1476,614,452,291
3,2025-06-23,333,133,133,133
4,2025-02-10,1534,646,482,305
5,2025-04-21,1468,635,478,305
6,2025-05-12,1515,594,438,295
7,2025-01-20,949,410,252,95
8,2025-05-26,1511,614,448,290
9,2025-02-03,1480,601,446,290


In [36]:
# Cálculo de porcentaje de retencion para cada semana

retencion_porcentajes = '''
WITH retencion AS (
    SELECT
        TO_CHAR(CAST(fecha_actividad AS DATE), 'MM-DD') AS cohorte,
        COUNT(*) AS usuarios_iniciales,
        COUNT(CASE WHEN dias_despues_registro >= 7  AND activo = 1 THEN 1 END) AS retenido_w1,
        COUNT(CASE WHEN dias_despues_registro >= 14 AND activo = 1 THEN 1 END) AS retenido_w2,
        COUNT(CASE WHEN dias_despues_registro >= 21 AND activo = 1 THEN 1 END) AS retenido_w3
    FROM user_activity
    GROUP BY cohorte
)

SELECT
    cohorte,
    usuarios_iniciales,
    retenido_w1,
    retenido_w2,
    retenido_w3,
    ROUND(retenido_w1::numeric / usuarios_iniciales, 2) AS semana_1,
    ROUND(retenido_w2::numeric / usuarios_iniciales, 2) AS semana_2,
    ROUND(retenido_w3::numeric / usuarios_iniciales, 2) AS semana_3
FROM retencion
ORDER BY cohorte;

'''
cohorte_final_porcentaje = pd.read_sql(retencion_porcentajes, con=engine)
cohorte_final_porcentaje

,cohorte,usuarios_iniciales,retenido_w1,retenido_w2,retenido_w3,semana_1,semana_2,semana_3
0,01-08,40,13,0,0,0.33,0.00,0.00
1,01-09,51,17,0,0,0.33,0.00,0.00
2,01-10,50,22,0,0,0.44,0.00,0.00
3,01-11,46,25,0,0,0.54,0.00,0.00
4,01-12,49,22,0,0,0.45,0.00,0.00
...,...,...,...,...,...,...,...,...
167,06-24,67,30,30,30,0.45,0.45,0.45
168,06-25,64,30,30,30,0.47,0.47,0.47
169,06-26,48,20,20,20,0.42,0.42,0.42
170,06-27,52,12,12,12,0.23,0.23,0.23


---

## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado** 
4. **Interpretar el resultado**  

---
Hipótesis estadística
   - **H₀ (Hipótesis nula):** No hay un impacto en la tasa de conversión de compra tras la modificación.
   - **H₁ (Hipótesis alternativa):** La tasa de conversión de compra es mayor tras la modificación.
   
**Test estadístico:** Prueba z  

**Nivel de significancia alpha:** 5%


In [37]:
# Carga de datos y librerías
from statsmodels.stats.proportion import proportions_ztest

df = pd.read_csv("experiment_checkout_ui.csv")

# Visualización de datos
df.head(5)

,id_usuario,variante,convirtio,dispositivo,pais,duracion_sesion,timestamp
0,exp_user_0,tratamiento,0,mobile,Argentina,114.41,2025-03-28
1,exp_user_1,tratamiento,0,desktop,Mexico,170.03,2025-01-15
2,exp_user_2,control,1,mobile,Colombia,140.21,2025-03-18
3,exp_user_3,tratamiento,0,mobile,Colombia,151.45,2025-06-03
4,exp_user_4,tratamiento,0,desktop,Mexico,299.96,2025-01-12


In [38]:
# Revisión de tipos de datos
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id_usuario       10000 non-null  object 
 1   variante         10000 non-null  object 
 2   convirtio        10000 non-null  int64  
 3   dispositivo      10000 non-null  object 
 4   pais             10000 non-null  object 
 5   duracion_sesion  10000 non-null  float64
 6   timestamp        10000 non-null  object 
dtypes: float64(1), int64(1), object(5)
memory usage: 547.0+ KB


In [39]:
# Cálculo de total de conversiones por variante
total_conversiones = df.groupby('variante')['convirtio'].sum()
total_conversiones

variante
control        779
tratamiento    820
Name: convirtio, dtype: int64

In [40]:
# Cálculo de total de usuarios por variante
total_usuarios = df.groupby('variante')['convirtio'].count()
total_usuarios

variante
control        4965
tratamiento    5035
Name: convirtio, dtype: int64

In [41]:
# Formato a lista de conversiones para la prueba z
exitos = [total_conversiones['control'], total_conversiones['tratamiento']]
exitos

[779, 820]

In [42]:
# Formato a lista de usuarios para la prueba z
observaciones = [total_usuarios['control'], total_usuarios['tratamiento']]
observaciones

[4965, 5035]

In [43]:
# Cálculo de prueba z 
zstat, p_value = proportions_ztest(exitos, observaciones)

print(f"Estadístico: {zstat}")
print(f"Valor p: {p_value}")

Estadístico: -0.8132782986429474
Valor p: 0.41605851639119995


In [44]:
# Umbral de significancia e interpretación de resultado

alpha = 0.05

if p_value < alpha:
    print("Rechazamos la hipótesis nula: Hay evidencia de una diferencia.")
else:
    print("No rechazamos la hipótesis nula: no hay evidencia suficiente de una diferencia")

No rechazamos la hipótesis nula: no hay evidencia suficiente de una diferencia


---

## 🔹 Paso 6: Comunicar los resultados (Dashboard en BI)

🎯 **Objetivo**:  
Crear un dashboard que muestre de manera clara y visual los resultados del análisis de ventas, costos, marketing y conversión. 

Se usarán los CSVs limpios del Paso 1:

- `orders_clean.csv`  
- `catalog_clean.csv`  
- `marketing_clean.csv`

---

1️⃣ Preparación de los datos
1. Cargar los CSVs en Power BI o Tableau.
2. Revisar relaciones:
   - `orders.nombre_producto` → `catalog.nombre_producto`
   - `orders.fecha_pedido` → tabla de fechas (crear calendario para análisis temporal)
   - `orders.fecha_pedido` → `dim_fecha.date`
3. Crear columnas calculadas necesarias
4. Crear tabla de fechas para poder calcular comparaciones YTD, YoY o períodos anteriores (`Previous Year`, `Previous Month`).

---

2️⃣ Dashboard 1: Overview Ejecutivo
**KPIs principales a mostrar:**
- Revenue total
- Profit total
- Gasto total en marketing
- Ticket promedio
- Cantidad promedio de productos por orden

**Visualizaciones sugeridas:**
- Tarjetas KPI para revenue, profit y gasto marketing
- Gráfico de líneas: evolución mensual de revenue o profit
- Gráfico de líneas YTD
- Gráfico de barras: revenue y profit por producto o categoría

---

 3️⃣ Dashboard 2: Detalle / Drill-through  
**Objetivo:** Permitir explorar los datos desde el KPI general hasta cada orden o producto.

**Visualizaciones sugeridas:**
- Tabla detallada de órdenes con:
  - producto, cantidad, revenue, cost, profit
  - color condicional (profit negativo en rojo, positivo en verde)
- Gráfico de barras por producto con medida `cantidad vendida`
- Drill-through: seleccionar un producto y ver todos los pedidos relacionados
- Filtros por fecha, categoría de producto, etc

---

## 🚀 Entrega Final

Comparte el acceso a tu Dashboard para revisión.   
Puedes entregar el Dashboard utilizando **Power BI o Tableau**.

Incluye **uno de los siguientes**:

- 🔗 Link público del dashboard publicado en **Power BI Service o Tableau Public / Tableau Cloud**
- 🔗 Link de **Google Drive o OneDrive** con el archivo del proyecto (`.pbix`) y los 3 csvs limpios.


### 📎 Enlace del Dashboard

In [45]:
# Link de Google Drive para revisión:
# https://drive.google.com/drive/folders/1DojSb_-SSIOWfuS0bywuEbYa6j6RlEIK?usp=drive_link


# link de power bi o tableau
# link de one drive / google drive